# Lab 3: LLM Workflows (10 min)

## Objectives
- Chain multiple model calls together
- Create text processing pipelines
- Use the output of one step as input for the next

## Concepts

### What is an LLM Workflow?
A pipeline where each step uses the model for a specific task:

```
Original text
    │
    ▼
[Step 1: Analysis]       → Understand the content
    │
    ▼
[Step 2: Transformation] → Process/transform
    │
    ▼
[Step 3: Output]         → Generate final result
```

Each step has its own specialized **system prompt**.

In [ ]:
# Setup
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

endpoint_base = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT", "").rstrip("/")
key = os.getenv("AZURE_AI_FOUNDRY_KEY", "")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

# For cognitiveservices.azure.com or openai.azure.com endpoints,
# the deployment must be embedded in the endpoint (don't pass model= in .complete())
if "cognitiveservices.azure.com" in endpoint_base or "openai.azure.com" in endpoint_base:
    endpoint = f"{endpoint_base}/openai/deployments/{model}"
    use_model_param = False
else:
    endpoint = endpoint_base if endpoint_base.endswith("/models") else endpoint_base + "/models"
    use_model_param = True

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
)

def execute_step(system_prompt: str, user_input: str, step_name: str) -> str:
    """Executes a workflow step."""
    print(f"\n⚙️ {step_name}...")
    kwargs = dict(
        messages=[
            SystemMessage(content=system_prompt),
            UserMessage(content=user_input),
        ],
        max_tokens=500,
        temperature=0.3,
    )
    if use_model_param:
        kwargs["model"] = model
    response = client.complete(**kwargs)
    result = response.choices[0].message.content
    print(f"   Tokens: {response.usage.total_tokens}")
    return result

print(f"✅ Setup complete — endpoint: {endpoint}")

✅ Setup completo — endpoint: https://foundry-mod2.cognitiveservices.azure.com/openai/deployments/gpt-4o


## 🖥️ Exercise 3.1: Text Processing Pipeline

Let's create a pipeline that:
1. **Analyzes** a technical text
2. **Simplifies** it for a non-technical audience
3. **Translates** it to English

In [ ]:
# Exercise 3.1: Text pipeline
original_text = """
Azure AI Foundry uses a hub-spoke architecture where the hub centralizes 
the management of shared resources like Azure AI Services, Key Vault and Storage Account,
while projects (spokes) inherit those resources and add specific configurations.
Models are deployed via serverless endpoints (MaaS) or managed compute,
and access is controlled by RBAC with managed identities for service-to-service authentication.
"""

print(f"📄 Original text ({len(original_text.split())} words)")

# Step 1: Analysis
analysis = execute_step(
    "Analyze the technical text and identify: 1) Main topic, 2) Key concepts, 3) Complexity level (1-5). Respond in structured format.",
    original_text,
    "Step 1: Analysis",
)
print(f"\n{analysis}")

# Step 2: Simplification
simplified = execute_step(
    "Rewrite the following technical text so that anyone without cloud computing knowledge can understand. Use everyday analogies. Maximum 3 sentences.",
    original_text,
    "Step 2: Simplification",
)
print(f"\n{simplified}")

# Step 3: Translation
translated = execute_step(
    "Translate the following text to Portuguese. Keep the simple and accessible tone.",
    simplified,
    "Step 3: Translation",
)
print(f"\n{translated}")

print("\n" + "=" * 60)
print("✅ Pipeline complete!")

📄 Texto original (61 palavras)

⚙️ Passo 1: Análise...


   Tokens: 384

**Análise do Texto Técnico**

1) **Tema Principal**:  
   Arquitetura e gestão de recursos no Azure AI Foundry para projetos de inteligência artificial.

2) **Conceitos-Chave**:  
   - **Azure AI Foundry**: Plataforma de IA no Azure.  
   - **Arquitetura Hub-Spoke**: Estrutura onde o hub centraliza recursos e os spokes (projetos) os utilizam com configurações específicas.  
   - **Recursos Partilhados**: Azure AI Services, Key Vault, Storage Account.  
   - **Deploy de Modelos**: Via endpoints serverless (MaaS - Model as a Service) ou managed compute.  
   - **Controle de Acesso**: RBAC (Role-Based Access Control) e managed identities para autenticação entre serviços.

3) **Nível de Complexidade**:  
   **4** (Avançado)  
   O texto apresenta conceitos técnicos relacionados à arquitetura de sistemas na nuvem, autenticação e deploy de modelos de IA, exigindo conhecimento prévio sobre Azure, gestão de recursos e segurança em ambientes de computação na nuvem.

⚙️ Passo 2: 

## 🖥️ Exercise 3.2: Feedback Analysis Pipeline

A more complex pipeline with sentiment analysis and categorization.

In [ ]:
# Exercise 3.2: Feedback analysis pipeline
import json

feedbacks = [
    "The Foundry portal is very intuitive, I loved the model deployment experience!",
    "It took me 3 hours to configure authentication. The documentation is confusing.",
    "The pricing is reasonable for enterprises, but startups need a larger free tier.",
    "The code interpreter in agents is fantastic for data analysis.",
]

# Step 1: Classify sentiment for each feedback
classification = execute_step(
    """Classify each feedback with:
- sentiment: positive/negative/neutral
- category: UX, documentation, pricing, functionality
- priority: high/medium/low
Respond in valid JSON as an array of objects.""",
    "\n".join([f"{i+1}. {f}" for i, f in enumerate(feedbacks)]),
    "Step 1: Classification",
)
print(f"\n{classification}")

# Step 2: Generate executive report
report = execute_step(
    "Based on the feedback classification, generate a 5-line executive report with: overall summary, strengths, areas for improvement, and a priority recommendation.",
    classification,
    "Step 2: Executive Report",
)
print(f"\n{report}")


⚙️ Passo 1: Classificação...
   Tokens: 341

```json
[
  {
    "feedback": "O portal do Foundry é muito intuitivo, adorei a experiência de deploy dos modelos!",
    "sentimento": "positivo",
    "categoria": "UX",
    "prioridade": "média"
  },
  {
    "feedback": "Demorei 3 horas a configurar a autenticação. A documentação é confusa.",
    "sentimento": "negativo",
    "categoria": "documentação",
    "prioridade": "alta"
  },
  {
    "feedback": "O preço é razoável para empresas, mas startups precisam de tier gratuito maior.",
    "sentimento": "neutro",
    "categoria": "preço",
    "prioridade": "média"
  },
  {
    "feedback": "O code interpreter nos agentes é fantástico para análise de dados.",
    "sentimento": "positivo",
    "categoria": "funcionalidade",
    "prioridade": "média"
  }
]
```

⚙️ Passo 2: Relatório Executivo...
   Tokens: 386

**Relatório Executivo:**

Resumo Geral: O feedback geral destaca uma experiência positiva com a usabilidade e funcionalidades do Foundry

## ✅ Summary

In this lab you learned:
- How to chain model calls into **pipelines**
- That each step can have a **specialized system prompt**
- How to use the output of one step as **input for the next**
- Patterns: analysis → transformation → output

**Next:** [Lab 4 - Multi-Agent Workflows](../lab04/lab04b-agent-workflows.ipynb)